In [1]:
!pip install transformers accelerate fastapi uvicorn pyngrok nest_asyncio

In [ ]:
import os
from pyngrok import ngrok

ngrok.set_auth_token(os.environ["AUTH_TOKEN"])

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

MODEL = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).cuda()
model.eval()

print(model.config)
print("Num layers:", len(model.model.layers))

In [ ]:
from transformers.cache_utils import DynamicCache

layers = model.model.layers
K = 4
print(f"Total layers: {len(layers)}, splitting at K={K}")

def _layer_hidden(out):
    return out[0] if isinstance(out, tuple) else out

def edge_forward(input_ids, cache, past_len=0):
    hidden = model.model.embed_tokens(input_ids)
    seq_len = hidden.shape[1]
    position_ids = torch.arange(past_len, past_len + seq_len, device=hidden.device).unsqueeze(0)
    pos_emb = model.model.rotary_emb(hidden, position_ids)

    for layer in layers[:K]:
        hidden = _layer_hidden(layer(
            hidden,
            position_ids=position_ids,
            past_key_values=cache,
            use_cache=True,
            position_embeddings=pos_emb,
        ))
    return hidden, position_ids

def cloud_forward(hidden, position_ids, cache):
    pos_emb = model.model.rotary_emb(hidden, position_ids)
    for layer in layers[K:]:
        hidden = _layer_hidden(layer(
            hidden,
            position_ids=position_ids,
            past_key_values=cache,
            use_cache=True,
            position_embeddings=pos_emb,
        ))
    hidden = model.model.norm(hidden)
    logits = model.lm_head(hidden)
    return logits

In [6]:
import asyncio
import torch
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok
from transformers.cache_utils import DynamicCache

app = FastAPI()

# session_id -> {"cache": DynamicCache, "past_len": int}
sessions = {}

class DecodeRequest(BaseModel):
    session_id: str
    hidden_state: list   # flattened floats
    shape: list           # e.g. [1, seq_len, hidden_dim]
    position_ids: list    # flattened position ids

@app.post("/session_start")
async def session_start(session_id: str):
    sessions[session_id] = {"cache": DynamicCache(), "past_len": 0}
    return {"status": "session created"}

@app.post("/decode")
async def decode(req: DecodeRequest):
    sess = sessions[req.session_id]
    hidden = torch.tensor(req.hidden_state, dtype=torch.float32, device="cuda").reshape(req.shape)
    pos_ids = torch.tensor(req.position_ids, dtype=torch.long, device="cuda").reshape(1, -1)

    with torch.no_grad():
        logits = cloud_forward(hidden, pos_ids, sess["cache"])
        next_token = logits[:, -1, :].argmax(-1, keepdim=True)

    sess["past_len"] += hidden.shape[1]
    return {"next_token": next_token.item()}

@app.get("/ping")
async def ping():
    return {"status": "alive"}


PORT = 8000

# Close any old ngrok tunnels
ngrok.kill()

config = uvicorn.Config(app=app, host="0.0.0.0", port=PORT, log_level="info")
server = uvicorn.Server(config)

server_task = asyncio.create_task(server.serve())

for _ in range(50):
    if server.started:
        break
    await asyncio.sleep(0.1)

if not server.started:
    raise RuntimeError("Uvicorn server failed to start.")

public_url = ngrok.connect(PORT).public_url

print("Public URL:", public_url)
print("Ping URL:", f"{public_url}/ping")

INFO:     Started server process [1875]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: https://liquid-cycling-kindle.ngrok-free.dev
Ping URL: https://liquid-cycling-kindle.ngrok-free.dev/ping
